In [ ]:

!pip install  openai matplotlib pillow pdfplumber openpyxl -q gradio pymupdf -q
print('✅ All dependencies installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.5 MB/s eta 0:00:00
✅ All dependencies installed!


In [ ]:

from openai import OpenAI

client = OpenAI(
    api_key="sk-lbvVTILP1EQ0LLgZwxyGqQ"
,
    base_url="https://apidev.navigatelabsai.com"
)
MODEL = "gpt-4.1-nano"

In [ ]:

import fitz
import gradio as gr

MODEL    = 'gpt-4.1-nano'
MAX_CHARS = 14000
notes    = {'text': '', 'filename': ''}

def llm(system, user, max_tokens=1200):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{'role': 'system', 'content': system},
                  {'role': 'user',   'content': user}],
    )
    return r.choices[0].message.content


# ── Upload ────────────────────────────────────────────────────
def handle_upload(file_obj):
    if file_obj is None:
        return '⚠️ No file detected.'
    path  = file_obj.name
    fname = path.split('/')[-1]
    if path.endswith('.pdf'):
        doc  = fitz.open(path)
        text = '\n'.join(p.get_text() for p in doc)
        doc.close()
    elif path.endswith('.txt'):
        with open(path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
    else:
        return '❌ Please upload a PDF or TXT file.'
    text = text[:MAX_CHARS]
    notes['text'] = text
    notes['filename'] = fname
    preview = text[:500].replace('\n', ' ') + '...'
    return (f"### ✅ '{fname}' loaded! ({len(text):,} chars)\n\n"
            f'**Preview:** {preview}\n\n---\n*All tabs are now ready.*')


# ── Summary ───────────────────────────────────────────────────
def summarise():
    if not notes['text']:
        return '⚠️ Upload your notes first.'
    sys = '''You are an expert academic tutor. Given the student notes produce:

## 📌 Topic Overview
3-5 sentences on what this topic is about.

## ✅ Key Revision Points
8-10 bullet points of the most important facts.

## 🔑 Important Terms
5 key terms with one-line definitions.

## ⚡ Common Exam Traps
2-3 misconceptions students often get wrong.

Be concise and student-friendly.'''
    return llm(sys, f"Notes:\n\n{notes['text']}")


# ── Q&A (multi-turn) ──────────────────────────────────────────
def answer_qa(user_msg, history):
    if not user_msg.strip():
        return history, ''
    context = (
        f"Student notes (primary source):\n\n{notes['text']}\n\n---"
        if notes['text'] else 'No notes uploaded. Use general knowledge.'
    )
    sys = ('You are a patient academic tutor. Answer clearly, reference the notes when relevant, '
           'show step-by-step reasoning for complex questions. Under 350 words.\n\n' + context)
    msgs = [{'role': 'system', 'content': sys}]
    for u, a in history:
        msgs += [{'role': 'user', 'content': u}, {'role': 'assistant', 'content': a}]
    msgs.append({'role': 'user', 'content': user_msg})
    r = client.chat.completions.create(model=MODEL, max_tokens=900, messages=msgs)
    reply = r.choices[0].message.content
    return history + [(user_msg, reply)], ''

def clear_chat():
    return [], ''


# ── Quiz ──────────────────────────────────────────────────────
def generate_quiz(num_q):
    if not notes['text']:
        return '⚠️ Upload your notes first.'
    sys = f'''Exam-setter: create exactly {int(num_q)} MCQ questions from the notes.
Format each as:

**Q1. [question]**
- A) ...
- B) ...
- C) ...
- D) ...

✅ **Answer: [letter] — [short explanation]**

---

Mix: 40% recall, 40% applied, 20% analytical. Cover different sections.'''
    return llm(sys, f"Notes:\n\n{notes['text']}", max_tokens=1400)


# ── Study Plan ────────────────────────────────────────────────
def make_study_plan(days, hours):
    ctx = f"Notes (first 3000 chars):\n{notes['text'][:3000]}" if notes['text'] else 'No notes — general plan.'
    sys = '''Study coach: create a day-by-day plan. Each day:
- 🗓️ Day + topic
- 📖 Activities (reading, practice, review)
- ⏱️ Time split
- 💡 Productivity tip
End with a motivational note. Use Markdown.'''
    return llm(sys, f'{ctx}\n\nDays: {int(days)} | Hours/day: {float(hours)}', max_tokens=1400)


# ── Gradio UI ─────────────────────────────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@300;400;600;700&display=swap');
body, .gradio-container { background:#0d0f14 !important; color:#e8eaf0 !important; font-family:'Sora',sans-serif !important; }
.hero { background:linear-gradient(135deg,#1a1f35,#0f172a,#1a0a2e); border:1px solid #2a3050; border-radius:16px; padding:28px 36px; margin-bottom:8px; }
.hero h1 { font-size:1.9rem; font-weight:700; background:linear-gradient(90deg,#a5b4fc,#818cf8,#c4b5fd); -webkit-background-clip:text; -webkit-text-fill-color:transparent; margin:0 0 6px; }
.hero p  { color:#94a3b8; font-size:0.92rem; margin:0; }
button.primary   { background:linear-gradient(135deg,#4f46e5,#7c3aed) !important; border:none !important; color:#fff !important; font-family:'Sora',sans-serif !important; font-weight:600 !important; border-radius:10px !important; padding:10px 22px !important; cursor:pointer !important; }
button.secondary { background:#1e2340 !important; border:1px solid #2e3560 !important; color:#94a3b8 !important; border-radius:10px !important; padding:8px 18px !important; cursor:pointer; }
textarea, input[type=text] { background:#1a1f2e !important; border:1px solid #2a3050 !important; border-radius:10px !important; color:#e8eaf0 !important; font-family:'Sora',sans-serif !important; }
textarea:focus, input[type=text]:focus { border-color:#6366f1 !important; box-shadow:0 0 0 3px rgba(99,102,241,.15) !important; outline:none !important; }
input[type=range] { accent-color:#6366f1 !important; }
"""

with gr.Blocks(css=CSS, title='Smart Study Notes Assistant') as demo:

    gr.HTML('<div class="hero"><h1>📚 Smart Study Notes Assistant</h1>'
            '<p>Upload your notes → summaries · Q&amp;A · quiz · study plan &nbsp;|&nbsp;</p></div>')

    with gr.Tab('📄  Upload Notes'):
        gr.Markdown('Upload a **PDF** or **TXT** file. It powers all other tabs.')
        file_in    = gr.File(label='Drop PDF or TXT here', file_types=['.pdf', '.txt'])
        upload_out = gr.Markdown('*Upload a file to get started.*')
        file_in.change(fn=handle_upload, inputs=file_in, outputs=upload_out)

    with gr.Tab('🗂️  Summary'):
        gr.Markdown('Structured summary: overview, revision bullets, key terms, exam traps.')
        sum_btn = gr.Button('✨  Generate Summary', variant='primary')
        sum_out = gr.Markdown()
        sum_btn.click(fn=summarise, inputs=[], outputs=sum_out)

    with gr.Tab('💬  Q & A'):
        gr.Markdown('Ask anything about your notes. Multi-turn memory keeps context across follow-ups.')
        chatbot = gr.Chatbot(label='Conversation', height=420, bubble_full_width=False)
        with gr.Row():
            qa_in     = gr.Textbox(placeholder='Ask a question…', lines=1, scale=8, show_label=False)
            send_btn  = gr.Button('Send ➤', variant='primary',   scale=1)
            clear_btn = gr.Button('Clear',  variant='secondary', scale=1)
        send_btn.click( fn=answer_qa,  inputs=[qa_in, chatbot], outputs=[chatbot, qa_in])
        qa_in.submit(   fn=answer_qa,  inputs=[qa_in, chatbot], outputs=[chatbot, qa_in])
        clear_btn.click(fn=clear_chat, inputs=[],               outputs=[chatbot, qa_in])

    with gr.Tab('🧠  Quiz'):
        gr.Markdown('Generate MCQ questions to test yourself before your exam.')
        num_q    = gr.Slider(3, 10, value=5, step=1, label='Number of questions')
        quiz_btn = gr.Button('🎲  Generate Quiz', variant='primary')
        quiz_out = gr.Markdown()
        quiz_btn.click(fn=generate_quiz, inputs=[num_q], outputs=quiz_out)

    with gr.Tab('📅  Study Plan'):
        gr.Markdown('Set your available time and get a personalised revision schedule.')
        with gr.Row():
            days_sl = gr.Slider(1, 30,  value=7,   step=1,   label='Days until exam')
            hrs_sl  = gr.Slider(0.5, 10, value=2,  step=0.5, label='Hours per day')
        plan_btn = gr.Button('📋  Build Study Plan', variant='primary')
        plan_out = gr.Markdown()
        plan_btn.click(fn=make_study_plan, inputs=[days_sl, hrs_sl], outputs=plan_out)

    gr.HTML('<p style="color:#374151;font-size:.78rem;text-align:center;padding-top:12px">Smart Study Notes Assistant</p>')

demo.launch(share=True)
print('\n🚀 Live! Open the public URL printed above.')

/tmp/ipykernel_5516/1048120915.py:131: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='Smart Study Notes Assistant') as demo:
/tmp/ipykernel_5516/1048120915.py:150: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='Conversation', height=420, bubble_full_width=False)
/tmp/ipykernel_5516/1048120915.py:150: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label='Conversation', height=420, bubble_full_width=False)
/tmp/ipykernel_5516/1048120915.py:150: DeprecationWarning: The 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdf42060bc2f1e905e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🚀 Live! Open the public URL printed above.
